<a href="https://colab.research.google.com/github/jovanfranco/usdmxn-volatilidad/blob/main/clase14_regresion_lineal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
from scipy import stats

# 1. El ejemplo a mano, con tres puntos
x = np.array([1, 2, 3.0])
y = np.array([2, 3, 5.0])
m = sm.OLS(y, sm.add_constant(x)).fit()
print("Constante y pendiente:", m.params.round(4))
print("Residuos:", m.resid.round(4))
print("R cuadrado:", round(float(m.rsquared), 4))
print("Error estándar:", m.bse.round(4))
print("t:", m.tvalues.round(3), "| p-valores:", m.pvalues.round(3))
print("Intervalo de 95%:")
print(m.conf_int().round(2))

# 2. Datos reales: USD/MXN contra USD/JPY
datos = yf.download(["MXN=X", "JPY=X"], start="2020-10-07", end="2026-10-22",
                    progress=False)["Close"]
rend = datos.pct_change().dropna()
X = sm.add_constant(rend["JPY=X"])

clasico = sm.OLS(rend["MXN=X"], X).fit()
robusto = sm.OLS(rend["MXN=X"], X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})

print("Constante y pendiente:", robusto.params.round(5).to_dict())
print("R cuadrado:", round(float(robusto.rsquared), 4))
print("Error estándar clásico:", clasico.bse.round(5).to_dict())
print("Error estándar robusto:", robusto.bse.round(5).to_dict())
print("t robusto:", robusto.tvalues.round(2).to_dict())
print("Intervalo de 95% robusto:")
print(robusto.conf_int().round(4))

# 3. Residuos: ¿se cumplen los supuestos?
res = clasico.resid
print("Autocorrelación de residuos:", round(float(res.autocorr(1)), 3))
print("Autocorrelación de residuos absolutos:", round(float(res.abs().autocorr(1)), 3))
print("Curtosis en exceso de los residuos:", round(float(stats.kurtosis(res)), 2))

# 4. Prueba con barajado: ¿qué t sale si no hay relación alguna?
rng = np.random.default_rng(42)
ts = []
for _ in range(500):
    x_bar = rng.permutation(rend["JPY=X"].values)
    mb = sm.OLS(rend["MXN=X"].values, sm.add_constant(x_bar)).fit()
    ts.append(mb.tvalues[1])
print("Proporción con |t| mayor a 2 al barajar:", round(float((np.abs(ts) > 2).mean()), 3))

Constante y pendiente: [0.3333 1.5   ]
Residuos: [ 0.1667 -0.3333  0.1667]
R cuadrado: 0.9643
Error estándar: [0.6236 0.2887]
t: [0.535 5.196] | p-valores: [0.687 0.121]
Intervalo de 95%:
[[-7.59  8.26]
 [-2.17  5.17]]


/tmp/ipykernel_6112/3166403828.py:20: FutureWarning: YF.download() has changed argument auto_adjust default to True
  datos = yf.download(["MXN=X", "JPY=X"], start="2020-10-07", end="2026-10-22",


Constante y pendiente: {'const': -0.00013, 'JPY=X': 0.16301}
R cuadrado: 0.0194
Error estándar clásico: {'const': 0.00017, 'JPY=X': 0.02927}
Error estándar robusto: {'const': 0.00017, 'JPY=X': 0.04571}
t robusto: {'const': -0.74, 'JPY=X': 3.57}
Intervalo de 95% robusto:
            0       1
const -0.0005  0.0002
JPY=X  0.0734  0.2526
Autocorrelación de residuos: -0.026
Autocorrelación de residuos absolutos: 0.204
Curtosis en exceso de los residuos: 3.84
Proporción con |t| mayor a 2 al barajar: 0.038
